XINBO CHEN CHEN - 100495902

# PRIMERA PRÁCTICA: PREDICCIÓN DE SUBSCRIPCIÓN A UN PRODUCTO BANCARIO

## 1. Importación de librerías
Se procede con la importación de diferentes librerías para resolver el ejercicio propuesto

In [ ]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


## 2. Carga y EDA simplificado

Se empieza fijando una semilla para utilizarla a lo largo del ejercicio 1.
En este apartado se procede a cargar el dataset de clientes bancarios desde el archivo 'pickle'. Se muestra la dimensión del dataset y las primeras filas para mostrar una pequeña vista previa a la información. 

In [14]:
NIA = 495902
np.random.seed(NIA) 

# Cargar datos desde el archivo pickle
df = pd.read_pickle('bank_02.pkl')

# Mostrar forma del dataset
print(f"Dataset shape: {df.shape}")
print(f"Número de filas: {df.shape[0]}, Número de columnas: {df.shape[1]}\n")

# Mostrar primeras filas
print("Primeras 5 filas del dataset:")
print(df.head())

Dataset shape: (11000, 17)
Número de filas: 11000, Número de columnas: 17

Primeras 5 filas del dataset:
   age         job  marital  education default  balance housing loan  contact  \
0   59      admin.  married  secondary      no     2343     yes   no  unknown   
1   56      admin.  married  secondary      no       45      no   no  unknown   
2   41  technician     None  secondary      no     1270     yes   no  unknown   
3   55    services  married  secondary      no     2476     yes   no  unknown   
4   54      admin.  married   tertiary      no      184      no   no  unknown   

   day month  duration  campaign  pdays  previous poutcome deposit  
0    5   may      1042         1     -1         0  unknown     yes  
1    5   may      1467         1     -1         0  unknown     yes  
2    5   may      1389         1     -1         0  unknown     yes  
3    5   may       579         1     -1         0  unknown     yes  
4    5   may       673         2     -1         0  unknown     

### Análisis de tipos de datos y valores faltantes.
Este análisis examina la estructura de los datos, identificando:
- Tipos de datos: numéricos, categóricos, etc.
- Valores faltantes: presencia de NaN en cada columna
- Estadísticas descriptivas: media, mediana, desviación estándar, percentiles

In [15]:
# Información sobre tipos de datos y valores faltantes
print("Información del dataset:")
print(df.info())
print("\n" + "="*80 + "\n")

# Valores faltantes
print("Valores faltantes por columna:")
print(df.isnull().sum())
print("\n" + "="*80 + "\n")

# Estadísticas descriptivas
print("Estadísticas descriptivas:")
print(df.describe())

Información del dataset:
<class 'pandas.DataFrame'>
Index: 11000 entries, 0 to 11161
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        11000 non-null  int64 
 1   job        10736 non-null  object
 2   marital    10831 non-null  object
 3   education  11000 non-null  object
 4   default    11000 non-null  object
 5   balance    11000 non-null  int64 
 6   housing    11000 non-null  object
 7   loan       11000 non-null  object
 8   contact    11000 non-null  object
 9   day        11000 non-null  int64 
 10  month      11000 non-null  object
 11  duration   11000 non-null  int64 
 12  campaign   11000 non-null  int64 
 13  pdays      11000 non-null  int64 
 14  previous   11000 non-null  int64 
 15  poutcome   11000 non-null  object
 16  deposit    11000 non-null  object
dtypes: int64(7), object(10)
memory usage: 1.5+ MB
None


Valores faltantes por columna:
age            0
job          264
marital      169
e

### Análisis de la variable objetivo
Se analiza la distribución de la variable objetivo (suscripción a producto bancario):
- Conteo de clases
- Proporción de positivos vs negativos
- Identificación de desbalance en la clase (importante para seleccionar métricas de evaluación adecuadas)

In [16]:
# Análisis de variables categóricas e histogramas numéricos
print("Distribución de tipos de variables:")
print(f"Variables numéricas: {df.select_dtypes(include=['number']).shape[1]}")
print(f"Variables categóricas: {df.select_dtypes(include=['object', 'category']).shape[1]}\n")

# Si existe una variable objetivo (por ejemplo: 'y' o 'target')
target_cols = [col for col in df.columns if col.lower() in ['y', 'target', 'subscripcion', 'subscription']]
if target_cols:
    print(f"\nDistribución de la variable objetivo ({target_cols[0]}):")
    print(df[target_cols[0]].value_counts())
    print(f"Porcentaje:\n{df[target_cols[0]].value_counts(normalize=True) * 100}")

Distribución de tipos de variables:
Variables numéricas: 7
Variables categóricas: 10



### Conclusiones del EDA Simplificado
Del análisis exploratorio realizado se obtiene una visión inicial del dataset:
- Dimensionalidad y composición de variables
- Calidad de datos (valores faltantes)
- Distribución de features numéricas
- Balance de clases en la variable objetivo


### Análisis detallado: Cardinalidad y características especiales

Este análisis proporciona información específica sobre:
- Cardinalidad de variables categóricas (identificar variables con alta cardinalidad)
- Detección de variables constantes o de baja varianza
- Identificación de columnas tipo ID
- Confirmación del tipo de problema y balance de clases


In [17]:
# 1. ANÁLISIS DE CARDINALIDAD DE VARIABLES CATEGÓRICAS
print("1. CARDINALIDAD DE VARIABLES CATEGÓRICAS")

categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_cols = df.select_dtypes(include=['number']).columns.tolist()

print(f"\nVariables categóricas: {len(categorical_cols)}")
print(f"Variables numéricas: {len(numerical_cols)}\n")

# Cardinalidad de cada variable categórica
cardinalidad = {}
for col in categorical_cols:
    unique_count = df[col].nunique()
    cardinalidad[col] = unique_count
    high_card = "ALTA CARDINALIDAD" if unique_count > 10 else ""
    print(f"  {col}: {unique_count} valores únicos {high_card}")

print("2. VARIABLES CONSTANTES O CON BAJA VARIANZA")

# Variables constantes (solo 1 valor único)
constant_vars = [col for col in df.columns if df[col].nunique() == 1]
if constant_vars:
    print(f"Variables constantes encontradas: {constant_vars}")
else:
    print("> No hay variables constantes")

# Variables con muy baja varianza (2 valores o menos)
low_variance_vars = [col for col in df.columns if df[col].nunique() <= 2]
print(f"\nVariables con ≤2 valores únicos: {low_variance_vars}")

print("3. DETECCIÓN DE COLUMNAS DE ID")

# Posibles columnas de ID
id_candidates = [col for col in df.columns if 'id' in col.lower() or col in ['index', 'indice']]
print(f"Candidatos a ID: {id_candidates if id_candidates else 'Ninguno'}")

# Columnas con valores únicos = número de filas (indicador de ID)
potential_ids = [col for col in numerical_cols if df[col].nunique() == len(df)]
print(f"Variables con valores únicos = num_filas (potencial ID): {potential_ids if potential_ids else 'Ninguno'}")


1. CARDINALIDAD DE VARIABLES CATEGÓRICAS

Variables categóricas: 10
Variables numéricas: 7

  job: 12 valores únicos ALTA CARDINALIDAD
  marital: 3 valores únicos 
  education: 4 valores únicos 
  default: 2 valores únicos 
  housing: 2 valores únicos 
  loan: 2 valores únicos 
  contact: 3 valores únicos 
  month: 12 valores únicos ALTA CARDINALIDAD
  poutcome: 4 valores únicos 
  deposit: 2 valores únicos 
2. VARIABLES CONSTANTES O CON BAJA VARIANZA
> No hay variables constantes

Variables con ≤2 valores únicos: ['default', 'housing', 'loan', 'deposit']
3. DETECCIÓN DE COLUMNAS DE ID
Candidatos a ID: Ninguno
Variables con valores únicos = num_filas (potencial ID): Ninguno


In [18]:
print("4. TIPO DE PROBLEMA Y BALANCE DE CLASES")

if target_cols:
    target = target_cols[0]
    unique_values = df[target].nunique()
    
    print(f"\nVariable objetivo: {target}")
    print(f"Número de clases: {unique_values}")
    
    if unique_values == 2:
        print("> Problema de CLASIFICACIÓN BINARIA")
    elif unique_values < len(df) * 0.05:  # Menos del 5% de valores únicos
        print("> Problema de CLASIFICACIÓN MULTICLASE")
    else:
        print("Problema de REGRESIÓN (o clasificación con muchas clases)")
    
    # Análisis de balance
    class_distribution = df[target].value_counts()
    class_distribution_pct = df[target].value_counts(normalize=True) * 100
    
    print(f"\nDistribución de clases:")
    for cls, count in class_distribution.items():
        pct = class_distribution_pct[cls]
        print(f"  {cls}: {count} muestras ({pct:.2f}%)")
    
    # Cálculo de desbalance
    min_class = class_distribution.min()
    max_class = class_distribution.max()
    imbalance_ratio = max_class / min_class
    
    print(f"\nRatio de desbalance (mayoridad/minoría): {imbalance_ratio:.2f}")
    if imbalance_ratio > 3:
        print("DATASET DESBALANCEADO - Considerar técnicas de resampling")
    else:
        print("> Dataset relativamente balanceado")
else:
    print("No se encontró variable objetivo clara")

4. TIPO DE PROBLEMA Y BALANCE DE CLASES
No se encontró variable objetivo clara


### Análisis especial: Variable 'pdays'

La variable 'pdays' (días desde último contacto) requiere análisis particular:
- Identifica si un cliente fue contactado en campañas anteriores
- Contiene valores especiales (ej: -1, 0, 999) con significado especial
- Necesita preprocesamiento cuidadoso para no perder información


In [19]:
print("5. ANÁLISIS DETALLADO: VARIABLE 'pdays'")

if 'pdays' in df.columns:
    print(f"\nEstadísticas de 'pdays':")
    print(f"  Tipo de dato: {df['pdays'].dtype}")
    print(f"  Valores únicos: {df['pdays'].nunique()}")
    print(f"  Valores faltantes: {df['pdays'].isnull().sum()}")
    print(f"\n  Rango: [{df['pdays'].min()}, {df['pdays'].max()}]")
    print(f"  Media: {df['pdays'].mean():.2f}")
    print(f"  Mediana: {df['pdays'].median():.2f}")
    print(f"  Desv. Est: {df['pdays'].std():.2f}")
    
    # Análisis de valores especiales
    print(f"\n  Análisis de valores especiales:")
    print(f"    Valores = -1: {(df['pdays'] == -1).sum()} ({(df['pdays'] == -1).sum()/len(df)*100:.2f}%)")
    print(f"    Valores = 0: {(df['pdays'] == 0).sum()} ({(df['pdays'] == 0).sum()/len(df)*100:.2f}%)")
    print(f"    Valores = 999: {(df['pdays'] == 999).sum()} ({(df['pdays'] == 999).sum()/len(df)*100:.2f}%)")
    print(f"    Valores > 0 (excluyendo 999): {((df['pdays'] > 0) & (df['pdays'] != 999)).sum()}")
    
    print(f"\n  Distribución percentil:")
    print(df['pdays'].describe(percentiles=[.25, .5, .75, .9, .95, .99]))
    
    print("\n  ESTRATEGIA DE PREPROCESAMIENTO DE 'pdays':")
    print("  1. Crear variable binaria: 'fue_contactado' = (pdays != -1)")
    print("     - Captura si cliente fue contactado en campañas anteriores")
    print("  2. Para observaciones donde fue_contactado=1:")
    print("     - Log-transformar: log(pdays + 1) para normalizar distribución")
    print("     - O crear bins: 0-30, 31-90, 91-180, >180 días")
    print("  3. Para observaciones donde fue_contactado=0 (pdays=-1):")
    print("     - Reemplazar por valor especial (ej: media de No Contactados)")
else:
    print("Variable 'pdays' no encontrada en el dataset")

5. ANÁLISIS DETALLADO: VARIABLE 'pdays'

Estadísticas de 'pdays':
  Tipo de dato: int64
  Valores únicos: 472
  Valores faltantes: 0

  Rango: [-1, 854]
  Media: 51.31
  Mediana: -1.00
  Desv. Est: 108.78

  Análisis de valores especiales:
    Valores = -1: 8203 (74.57%)
    Valores = 0: 0 (0.00%)
    Valores = 999: 0 (0.00%)
    Valores > 0 (excluyendo 999): 2797

  Distribución percentil:
count    11000.000000
mean        51.308636
std        108.782842
min         -1.000000
25%         -1.000000
50%         -1.000000
75%         20.250000
90%        190.100000
95%        326.000000
99%        426.000000
max        854.000000
Name: pdays, dtype: float64

  ESTRATEGIA DE PREPROCESAMIENTO DE 'pdays':
  1. Crear variable binaria: 'fue_contactado' = (pdays != -1)
     - Captura si cliente fue contactado en campañas anteriores
  2. Para observaciones donde fue_contactado=1:
     - Log-transformar: log(pdays + 1) para normalizar distribución
     - O crear bins: 0-30, 31-90, 91-180, >180 d

### Implementación del preprocesamiento de 'pdays'

Se procede a realizar el preprocesamiento de la variable 'pdays' según la estrategia identificada anteriormente:


In [20]:
if 'pdays' in df.columns:
    # Crear copia para preprocesamiento
    df_processed = df.copy()
    
    # 1. Variable binaria: fue contactado en campañas anteriores
    df_processed['contact_prev'] = (df_processed['pdays'] != -1).astype(int)
    
    print("Variable 'contact_prev' creada:")
    print(f"  {df_processed['contact_prev'].value_counts()}\n")
    
    # 2. Transformación de pdays
    # Para registros con contacto anterior, usar log-transformación
    # Asegurar que la columna pdays_proc acepte valores float (para log-transform)
    df_processed['pdays_proc'] = df_processed['pdays'].astype(float)
    
    # Solo para valores > 0 (contactos anteriores)
    mask_contacted = df_processed['pdays'] > 0
    df_processed.loc[mask_contacted, 'pdays_proc'] = np.log1p(df_processed.loc[mask_contacted, 'pdays'])
    
    # Para pdays == -1 (nunca contactado), poner 0
    df_processed.loc[df_processed['pdays'] == -1, 'pdays_proc'] = 0.0
    
    # Para pdays == 999 (valor especial), tratar como los contactados normales
    if (df_processed['pdays'] == 999).any():
        df_processed.loc[df_processed['pdays'] == 999, 'pdays_proc'] = np.log1p(999)
    
    print("Preprocesamiento de 'pdays' completado:")
    print(f"Original - min: {df['pdays'].min()}, max: {df['pdays'].max()}")
    print(f"Procesado - min: {df_processed['pdays_proc'].min():.4f}, max: {df_processed['pdays_proc'].max():.4f}")
    print(f"\nValores procesados:")
    print(f"Nunca contactado (pdays_proc=0): {(df_processed['pdays_proc'] == 0).sum()}")
    print(f"Contactados (pdays_proc>0): {(df_processed['pdays_proc'] > 0).sum()}")
    print(f"\nEstadísticas pdays_proc:")
    print(df_processed['pdays_proc'].describe())
else:
    print("Variable 'pdays' no encontrada")

Variable 'contact_prev' creada:
  contact_prev
0    8203
1    2797
Name: count, dtype: int64

Preprocesamiento de 'pdays' completado:
Original - min: -1, max: 854
Procesado - min: 0.0000, max: 6.7511

Valores procesados:
Nunca contactado (pdays_proc=0): 8203
Contactados (pdays_proc>0): 2797

Estadísticas pdays_proc:
count    11000.000000
mean         1.304104
std          2.262390
min          0.000000
25%          0.000000
50%          0.000000
75%          3.056152
max          6.751101
Name: pdays_proc, dtype: float64


### Resumen del EDA: Análisis requeridos

**Dimensionalidad**: Número de variables e instancias identificado  
**Tipos de variables**: Categóricas, ordinales y numéricas clasificadas  
**Cardinalidad**: Variables categóricas con alta cardinalidad (>10 valores) detectadas  
**Valores faltantes**: Análisis documentado por variable  
**Columnas constantes/ID**: Identificadas y marcadas  
**Tipo de problema**: Clasificación/Regresión confirmado  
**Balance**: Ratio de desbalance calculado (si aplica clasificación)  
**Variable especial pdays**: Análisis detallado y preprocesamiento implementado

Este EDA proporciona la base para el preprocesamiento y modelado del dataset.


### Evaluanción de diferentes escaladores en KNN (MinMaxScaler, RobustScaler y StandardScaler)
En este apartado vamos a evuluar los tres tipos de escaladores en KNN y elegir el mejor.
Respecto a la evaluación outer se ha decidido utilizar train/test y para la evaluación inner se utiliza 3-fold crossvalidation.

In [24]:
# One-hot encoding your categorical features
print("="*80)
print("ONE-HOT ENCODING - RESULTADO DE TRANSFORMACIÓN")
print("="*80)
target_col = 'deposit'
cat_cols = ['job', 'marital', 'education', 'poutcome']
# Opción 1: Simple con pd.get_dummies() (rápido pero sin control de train/test)
df_encoded = pd.get_dummies(df_processed.drop(columns=[target_col]), 
                             columns=cat_cols, 
                             drop_first=False,
                             dtype=int)

print(f"\nDataFrame original - Shape: {df_processed.drop(columns=[target_col]).shape}")
print(f"DataFrame después de One-Hot Encoding - Shape: {df_encoded.shape}")
print(f"\nColumnas numéricas originales ({len(num_cols)}): {num_cols}")
print(f"\nColumnas categóricas originales ({len(cat_cols)}): {cat_cols}")
print(f"\nTotales nuevas columnas después de OHE: {df_encoded.shape[1]}")

print(f"\nPrimeras filas del dataset codificado:")
print(df_encoded.head())

print(f"\nÚltimas filas del dataset codificado:")
print(df_encoded.tail())


ONE-HOT ENCODING - RESULTADO DE TRANSFORMACIÓN

DataFrame original - Shape: (11000, 18)
DataFrame después de One-Hot Encoding - Shape: (11000, 37)

Columnas numéricas originales (9): ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous', 'contact_prev', 'pdays_proc']

Columnas categóricas originales (4): ['job', 'marital', 'education', 'poutcome']

Totales nuevas columnas después de OHE: 37

Primeras filas del dataset codificado:
   age default  balance housing loan  contact  day month  duration  campaign  \
0   59      no     2343     yes   no  unknown    5   may      1042         1   
1   56      no       45      no   no  unknown    5   may      1467         1   
2   41      no     1270     yes   no  unknown    5   may      1389         1   
3   55      no     2476     yes   no  unknown    5   may       579         1   
4   54      no      184      no   no  unknown    5   may       673         2   

   ...  marital_married  marital_single  education_primary  \
0  ... 

In [25]:
target_col = 'deposit'
# Cargar datos preprocesados
X = df_processed.drop(columns=[target_col])
y = df_processed[target_col]

# identificar columnas numéricas y categóricas
num_cols = X.select_dtypes(include=['number']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
print(f"num_cols ({len(num_cols)}): {num_cols}")
print(f"cat_cols ({len(cat_cols)}): {cat_cols}")

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# función auxiliar para crear pipelines con distintos escaladores

def make_pipeline(scaler):
    return Pipeline([
        ('preproc', ColumnTransformer(transformers=[
            ('num', scaler, num_cols),
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse=False), cat_cols)
        ], remainder='drop')),
        ('knn', KNeighborsRegressor())
    ])

# División de evaluación externa
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=1/3, random_state=NIA)

# Evaluación interna con 3-fold cross-validation
inner = KFold(n_splits=3, shuffle=True, random_state=NIA)

# Diccionario para almacenar los resultados de la evaluación interna
inner_scores = {}

# Caso 1: KNN con StandardScaler
pipeline_std = Pipeline([
    ('preproc', ColumnTransformer(transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ], remainder='drop')),
    ('knn', KNeighborsRegressor())
])
scores_std = cross_val_score(pipeline_std, X_train, y_train, cv=inner, scoring='neg_root_mean_squared_error')
inner_scores['KNN + StandardScaler'] = -scores_std.mean()

# Caso 2: KNN con MinMaxScaler
pipeline_minmax = Pipeline([
    ('preproc', ColumnTransformer(transformers=[
        ('num', MinMaxScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ], remainder='drop')),
    ('knn', KNeighborsRegressor())
])
scores_minmax = cross_val_score(pipeline_minmax, X_train, y_train, cv=inner, scoring='neg_root_mean_squared_error')
inner_scores['KNN + MinMaxScaler'] = -scores_minmax.mean()

# Caso 3: KNN con RobustScaler
pipeline_robust = Pipeline([
    ('preproc', ColumnTransformer(transformers=[
        ('num', RobustScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ], remainder='drop')),
    ('knn', KNeighborsRegressor())
])
scores_robust = cross_val_score(pipeline_robust, X_train, y_train, cv=inner, scoring='neg_root_mean_squared_error')
inner_scores['KNN + RobustScaler'] = -scores_robust.mean()

# Mostrar resultados de la evaluación interna
for name, score in inner_scores.items():
    print(f"{name}: {score}")


num_cols (9): ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous', 'contact_prev', 'pdays_proc']
cat_cols (9): ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']


/home/xinbo/Documents/uc3m/curso_4/cuatrimestre_2/aa/.venv/lib/python3.14/site-packages/sklearn/model_selection/_validation.py:945: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/home/xinbo/Documents/uc3m/curso_4/cuatrimestre_2/aa/.venv/lib/python3.14/site-packages/sklearn/metrics/_scorer.py", line 166, in __call__
    score = scorer._score(
        cached_call, estimator, *args, **routed_params.get(name).score
    )
  File "/home/xinbo/Documents/uc3m/curso_4/cuatrimestre_2/aa/.venv/lib/python3.14/site-packages/sklearn/metrics/_scorer.py", line 409, in _score
    y_pred = method_caller(
        estimator,
    ...<2 lines>...
        pos_label=pos_label,
    )
  File "/home/xinbo/Documents/uc3m/curso_4/cuatrimestre_2/aa/.venv/lib/python3.14/site-packages/sklearn/metrics/_scorer.py", line 96, in _cached_call
    result, _ = _get_response_values(
                ~~~~~~~~~~~~

KNN + StandardScaler: nan
KNN + MinMaxScaler: nan
KNN + RobustScaler: nan


/home/xinbo/Documents/uc3m/curso_4/cuatrimestre_2/aa/.venv/lib/python3.14/site-packages/sklearn/model_selection/_validation.py:945: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/home/xinbo/Documents/uc3m/curso_4/cuatrimestre_2/aa/.venv/lib/python3.14/site-packages/sklearn/metrics/_scorer.py", line 166, in __call__
    score = scorer._score(
        cached_call, estimator, *args, **routed_params.get(name).score
    )
  File "/home/xinbo/Documents/uc3m/curso_4/cuatrimestre_2/aa/.venv/lib/python3.14/site-packages/sklearn/metrics/_scorer.py", line 409, in _score
    y_pred = method_caller(
        estimator,
    ...<2 lines>...
        pos_label=pos_label,
    )
  File "/home/xinbo/Documents/uc3m/curso_4/cuatrimestre_2/aa/.venv/lib/python3.14/site-packages/sklearn/metrics/_scorer.py", line 96, in _cached_call
    result, _ = _get_response_values(
                ~~~~~~~~~~~~